In [ ]:
#for better result print and solve the case for no result
def solve_subset_sum(xs, t, shots=500, attempts=6, iters_list=None, dominance_threshold=0.20):
    """
    Returns:
      - (I, info_dict) if a correct solution is found (verified classically).
      - (None, info_dict) if no solution is found after several attempts
        (interpreted as "no solution with high probability").

    Strategy:
      - Try several Grover iteration counts (Grover oscillates).
      - Each time: run sample_solutions -> get candidate I.
      - Verify classically: sum(xs[i] for i in I) == t.
      - If none verify, return None.
    """
    n = len(xs)
    if n == 0:
        return None, {"reason": "empty input"}

    # Default schedule: try around the usual Grover heuristic sqrt(2^n)
    if iters_list is None:
        base = max(1, int((math.pi / 4) * math.sqrt(2 ** n)))
        iters_list = sorted(set([1, 2, 3, max(1, base - 1), base, base + 1, max(1, base // 2)]))
        iters_list = iters_list[:attempts]

    best_seen = None  # store best dominance attempt for reporting

    for iters in iters_list:
        counts, best, s_bits, I = sample_solutions(xs, t, shots=shots, iters=iters)
        candidate_sum = sum(xs[i] for i in I)

        dominance = counts[best] / shots
        if best_seen is None or dominance > best_seen["dominance"]:
            best_seen = {
                "iters": iters,
                "best": best,
                "s_bits": s_bits,
                "I": I,
                "dominance": dominance,
                "candidate_sum": candidate_sum,
            }

        # Verified success
        if candidate_sum == t:
            return I, {
                "status": "FOUND",
                "iters": iters,
                "best_bitstring": best,
                "dominance": dominance,
                "s_bits": s_bits,
                "I": I,
                "candidate_sum": candidate_sum,
                "shots": shots,
            }

    # Nothing verified => "no solution with high probability"
    # dominance_threshold is just an extra signal; we still return None regardless.
    return None, {
        "status": "NO_SOLUTION_WHP",
        "shots": shots,
        "attempts": len(iters_list),
        "iters_list": iters_list,
        "best_seen": best_seen,
        "note": (
            "No measured candidate verified. If no true solution exists, Grover cannot amplify, "
            "so outcomes remain near-uniform; repeated failures support 'no solution' conclusion."
        ),
    }

In [ ]:
if __name__ == "__main__":
    xs = [3, 5, 6, 2]
    t = 11

    I, info = solve_subset_sum(xs, t, shots=500, attempts=6)

    print("INFO:", info)
    if I is None:
        print("No solution found (high probability).")
    else:
        print("Solution I:", I)
        print("Check sum:", sum(xs[i] for i in I))